# 📊 Progetto Trasversale — Data Analysis

**Nome e cognome**: Andrea Pavan  
**Dataset scelto**: Sample Superstore (E-commerce)  
**Link Kaggle**: https://www.kaggle.com/datasets/vivek468/superstore-dataset-final  
**Perché ho scelto questo dataset**: Dataset reale di vendite e-commerce con informazioni su ordini, clienti, prodotti e profitti. Permette di ragionare come un analista di business e scoprire dove un'azienda guadagna (e perde) davvero.

---

### 📋 Le 5 domande di ricerca

1. **Q1** — Quanti ordini, prodotti, clienti, regioni e categorie ci sono? Su che periodo si estendono i dati?
2. **Q2** — Quali categorie e sotto-categorie generano più vendite e più profitti? Ci sono sotto-categorie in perdita?
3. **Q3** — Come variano le vendite mese per mese? Si vede una stagionalità? Qual è il mese migliore e quello peggiore?
4. **Q4** — I 3 segmenti di clienti (Consumer, Corporate, Home Office) sono ugualmente profittevoli? Chi compra di più, chi rende di più?
5. **Q5** — Lo sconto fa bene o male all'azienda? Esiste una soglia di sconto oltre la quale il profitto diventa negativo?

**Bonus (opzionale):** Principio di Pareto — quanti prodotti (in percentuale) generano l'80% del profitto totale?

---

> 💡 **Come leggere questo notebook**: ogni sezione risponde a una domanda. Per ogni domanda trovi il codice, il risultato e un commento in testo che interpreta i numeri.

## ⚙️ Setup — Importazione librerie

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 11

print("Setup completato ✓")

## 📥 Caricamento del dataset

In [ ]:
df = pd.read_csv("Sample - Superstore.csv", encoding="latin-1")
print(f"Dataset caricato: {df.shape[0]} righe × {df.shape[1]} colonne")
df.head()

## 🔍 Q1 — Esplorazione iniziale del dataset

> **Domanda Q1**: Quanti ordini, prodotti, clienti, regioni e categorie ci sono? Su che periodo si estendono i dati?

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print(f"Ordini totali:       {df['Order ID'].nunique()}")
print(f"Prodotti unici:      {df['Product ID'].nunique()}")
print(f"Clienti unici:       {df['Customer ID'].nunique()}")
print(f"Regioni:             {df['Region'].nunique()} → {list(df['Region'].unique())}")
print(f"Categorie:           {df['Category'].nunique()} → {list(df['Category'].unique())}")
print(f"Sotto-categorie:     {df['Sub-Category'].nunique()}")
print(f"\nPeriodo: da {df['Order Date'].min()} a {df['Order Date'].max()}")

**📝 Risposta alla Q1:**

_[Scrivi qui in 3-5 righe cosa hai scoperto: numero di ordini, prodotti, clienti, regioni, categorie e il periodo temporale dei dati.]_

## 🧹 Pulizia del dataset

In [ ]:
# Valori mancanti per colonna
df.isna().sum()

In [ ]:
# Duplicati
print(f"Righe duplicate: {df.duplicated().sum()}")

In [ ]:
# Conversione colonne data da stringa a datetime
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])

# Verifica
print(df[['Order Date', 'Ship Date']].dtypes)
print(f"\nRighe dopo la pulizia: {df.shape[0]}")

**📝 Pulizie effettuate:**

- Convertite le colonne `Order Date` e `Ship Date` da stringa a `datetime` — necessario per l'analisi temporale della Q3
- Nessun valore mancante trovato
- Nessuna riga duplicata trovata

## 📊 Q2 — Vendite e profitti per categoria e sotto-categoria

> **Domanda Q2**: Quali categorie e sotto-categorie generano più vendite e più profitti? Ci sono sotto-categorie in perdita?

In [ ]:
# Vendite e profitti per categoria
q2_cat = df.groupby('Category')[['Sales', 'Profit']].sum().sort_values('Profit', ascending=False)
q2_cat['Profit Margin %'] = (q2_cat['Profit'] / q2_cat['Sales'] * 100).round(1)
q2_cat

In [ ]:
# Profitto per sotto-categoria (evidenzia quelle in perdita)
q2_sub = df.groupby('Sub-Category')['Profit'].sum().sort_values()
colori = ['#e74c3c' if x < 0 else '#2ecc71' for x in q2_sub]

q2_sub.plot(kind='barh', color=colori, figsize=(10, 7))
plt.axvline(x=0, color='black', linewidth=0.8)
plt.title('Profitto per sotto-categoria (rosso = in perdita)')
plt.xlabel('Profitto totale ($)')
plt.tight_layout()
plt.show()

**💡 Risposta alla Q2:**

_[Scrivi qui quali categorie guadagnano di più, quali sotto-categorie sono in perdita e perché è importante saperlo.]_

## 📈 Q3 — Analisi temporale delle vendite

> **Domanda Q3**: Come variano le vendite mese per mese? Si vede una stagionalità? Qual è il mese migliore e quello peggiore?

In [ ]:
# Estrazione mese e anno
df['Year'] = df['Order Date'].dt.year
df['Month'] = df['Order Date'].dt.month

# Vendite mensili aggregate su tutti gli anni
vendite_mensili = df.groupby('Month')['Sales'].sum()
nomi_mesi = ['Gen','Feb','Mar','Apr','Mag','Giu','Lug','Ago','Set','Ott','Nov','Dic']
vendite_mensili.index = nomi_mesi

vendite_mensili.plot(kind='bar', color='#3498db')
plt.title('Vendite totali per mese (tutti gli anni)')
plt.xlabel('Mese')
plt.ylabel('Vendite ($)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print(f"Mese migliore:  {vendite_mensili.idxmax()} (${vendite_mensili.max():,.0f})")
print(f"Mese peggiore: {vendite_mensili.idxmin()} (${vendite_mensili.min():,.0f})")

In [ ]:
# Andamento annuale
vendite_annuali = df.groupby('Year')['Sales'].sum()
vendite_annuali.plot(kind='line', marker='o', color='#2ecc71')
plt.title('Vendite totali per anno')
plt.xlabel('Anno')
plt.ylabel('Vendite ($)')
plt.tight_layout()
plt.show()

**💡 Risposta alla Q3:**

_[Scrivi qui qual è il mese migliore e peggiore, se si vede una stagionalità (es. picchi a fine anno?) e come crescono le vendite anno per anno.]_

## 🔀 Q4 — Confronto tra segmenti di clienti

> **Domanda Q4**: I 3 segmenti di clienti (Consumer, Corporate, Home Office) sono ugualmente profittevoli? Chi compra di più, chi rende di più?

In [ ]:
# Riepilogo per segmento
q4 = df.groupby('Segment').agg(
    Ordini=('Order ID', 'nunique'),
    Vendite_totali=('Sales', 'sum'),
    Profitto_totale=('Profit', 'sum'),
    Profitto_medio=('Profit', 'mean')
).round(2)
q4['Margin %'] = (q4['Profitto_totale'] / q4['Vendite_totali'] * 100).round(1)
q4.sort_values('Profitto_totale', ascending=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

df.groupby('Segment')['Sales'].sum().plot(kind='bar', ax=axes[0], color='#3498db')
axes[0].set_title('Vendite totali per segmento')
axes[0].set_ylabel('Vendite ($)')
axes[0].tick_params(axis='x', rotation=0)

df.groupby('Segment')['Profit'].sum().plot(kind='bar', ax=axes[1], color='#2ecc71')
axes[1].set_title('Profitto totale per segmento')
axes[1].set_ylabel('Profitto ($)')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

**💡 Risposta alla Q4:**

_[Scrivi qui quale segmento compra di più, quale rende di più, e se ci sono differenze significative nel margine tra i segmenti.]_

## 🤔 Q5 — Lo sconto fa bene o male all'azienda?

> **Domanda Q5**: Lo sconto fa bene o male all'azienda? Esiste una soglia di sconto oltre la quale il profitto diventa negativo?

In [ ]:
# Correlazione sconto-profitto
print("Correlazione Discount → Profit:")
print(df[['Discount', 'Profit']].corr())

In [ ]:
# Scatter plot sconto vs profitto
plt.scatter(df['Discount'], df['Profit'], alpha=0.3, color='#e74c3c')
plt.axhline(y=0, color='black', linestyle='--', linewidth=1)
plt.title('Sconto vs Profitto')
plt.xlabel('Sconto applicato')
plt.ylabel('Profitto ($)')
plt.tight_layout()
plt.show()

In [ ]:
# Profitto medio per fascia di sconto
bins = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
labels = ['0-10%','10-20%','20-30%','30-40%','40-50%','50-60%','60-70%','70-80%','80-90%']
df['Fascia_Sconto'] = pd.cut(df['Discount'], bins=bins, labels=labels)

profitto_per_fascia = df.groupby('Fascia_Sconto', observed=True)['Profit'].mean()
colori = ['#e74c3c' if x < 0 else '#2ecc71' for x in profitto_per_fascia]

profitto_per_fascia.plot(kind='bar', color=colori)
plt.axhline(y=0, color='black', linewidth=0.8)
plt.title('Profitto medio per fascia di sconto')
plt.xlabel('Fascia di sconto')
plt.ylabel('Profitto medio ($)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

# Perdite totali con sconto >= 40%
perdite = df[df['Discount'] >= 0.4]['Profit'].sum()
print(f"Perdite totali con sconto ≥ 40%: ${perdite:,.0f}")

**💡 Risposta alla Q5:**

_[Scrivi qui: qual è la soglia di sconto critica? Quanto perde l'azienda con sconti eccessivi? Correlazione positiva o negativa? Attenzione: correlazione non è causalità.]_

> ⚠️ **Nota**: correlazione non significa causalità. Potrebbe essere che i prodotti difficili da vendere ricevano più sconti, e non che lo sconto stesso causi la perdita.

## 🎯 Conclusioni e insight chiave

---

**🥇 Insight 1:** _[Il più sorprendente — spiegalo in 2-3 righe.]_

**🥈 Insight 2:** _[Il secondo più interessante.]_

**🥉 Insight 3:** _[Il terzo.]_

---

**Grafico killer per la presentazione:** _[Scrivi qui quale grafico userai nella presentazione finale.]_

## 🏆 Bonus — Principio di Pareto (80/20)

> Quanti prodotti (in percentuale) generano l'80% del profitto totale?

In [ ]:
# Profitto per prodotto, ordinato e cumulato
profitto_prodotti = df.groupby('Product Name')['Profit'].sum().sort_values(ascending=False)
profitto_prodotti = profitto_prodotti[profitto_prodotti > 0]  # solo prodotti in positivo

profitto_cum = profitto_prodotti.cumsum() / profitto_prodotti.sum() * 100
n_80 = (profitto_cum <= 80).sum()
perc_prodotti = n_80 / len(profitto_prodotti) * 100

print(f"Prodotti che generano l'80% del profitto: {n_80} su {len(profitto_prodotti)} ({perc_prodotti:.1f}%)")

# Grafico cumulativo
plt.plot(range(len(profitto_cum)), profitto_cum.values, color='#3498db')
plt.axhline(y=80, color='red', linestyle='--', label='80%')
plt.axvline(x=n_80, color='orange', linestyle='--', label=f'{n_80} prodotti')
plt.title('Curva di Pareto — Profitto cumulativo per prodotto')
plt.xlabel('Numero di prodotti (dal più redditizio)')
plt.ylabel('% profitto cumulativo')
plt.legend()
plt.tight_layout()
plt.show()

**💡 Risposta al Bonus:**

_[Scrivi qui se la regola 80/20 si conferma e cosa significa praticamente per l'azienda.]_

---

🎉 **Fine del progetto!** Checklist finale:
- ☐ Nome e cognome compilati in cima
- ☐ Tutte le 5 domande hanno risposta in testo Markdown
- ☐ Ogni grafico ha titolo ed etichette
- ☐ Il notebook si esegue dall'inizio alla fine senza errori